In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=1000, n_features=25, n_informative=5, n_redundant=10, random_state=42)
feature_names = [f'Feature_{i}' for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)

X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.3, random_state=42)

estimator = RandomForestClassifier(n_estimators=100, random_state=42)
selector = RFE(estimator, n_features_to_select=5, step=1)
selector = selector.fit(X_train, y_train)

selected_features = df.columns[selector.support_]
print("Selected Features:")
print(selected_features.tolist())

ranking = pd.DataFrame({'Feature': feature_names, 'Ranking': selector.ranking_})
print("\nFeature Rankings (1 is selected):")
print(ranking.sort_values(by='Ranking'))

X_train_selected = selector.transform(X_train)
X_test_selected = selector.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_selected, y_train)
y_pred = model.predict(X_test_selected)

print(f"\nAccuracy with selected features: {accuracy_score(y_test, y_pred):.4f}")

importances = selector.estimator_.feature_importances_
plt.figure(figsize=(10, 6))
plt.barh(selected_features, importances)
plt.xlabel("Feature Importance")
plt.ylabel("Selected Features")
plt.title("Importance of RFE Selected Features")
plt.show()